In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!mkdir -p h_and_m

In [ ]:
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations \
 -f articles.csv -p h_and_m

In [ ]:
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations \
 -f customers.csv -p h_and_m

In [ ]:
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations \
 -f transactions_train.csv -p h_and_m

In [ ]:
!ls -lh h_and_m

In [ ]:
!unzip -o h_and_m/articles.csv.zip -d h_and_m
!unzip -o h_and_m/customers.csv.zip -d h_and_m
!unzip -o h_and_m/transactions_train.csv.zip -d h_and_m

In [ ]:
!ls -lh h_and_m/*.csv

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

In [ ]:
arti = spark.read.csv("h_and_m/articles.csv", header=True, inferSchema=True)
cust = spark.read.csv("h_and_m/customers.csv", header=True, inferSchema=True)
tran = spark.read.csv("h_and_m/transactions_train.csv", header=True, inferSchema=True)


In [ ]:
print('===articles===')
print(arti.count())
len(arti.columns)

In [ ]:
arti.show(7)

In [ ]:
arti.printSchema()

In [ ]:
print('===custmer===')
print(cust.count())
len(cust.columns)

In [ ]:
cust.show(7)

In [ ]:
cust.printSchema()

In [ ]:
print('===transactions_train===')
print(tran.count())
len(tran.columns)

In [ ]:
tran.show(7)

In [ ]:
tran.printSchema()

# **Cheacking The Null Values**

In [ ]:
from pyspark.sql.functions import col,sum,when
arti.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in arti.columns
]).show()

cust.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in cust.columns
]).show()

tran.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in tran.columns
]).show()

In [ ]:
cust.count()

In [ ]:
for column in cust.columns:
    print(column, ":", cust.select(column).distinct().count())

In [ ]:
from pyspark.sql.functions import col, when

cust = cust.withColumn("FN",when(col("FN").isNull(), 0.0).otherwise(1.0)) \
           .withColumn('Active',when(col('Active').isNull(), 0.0).otherwise(1.0))

In [ ]:
cust = cust.withColumn("club_member_status",when(col("club_member_status").isNull(), 'Unknown').otherwise(col("club_member_status"))) \
           .withColumn("fashion_news_frequency",when(col("fashion_news_frequency").isNull(), 'Unknown').otherwise(col("fashion_news_frequency")))

In [ ]:
cust.show(10)

In [ ]:
cust.select('age').summary().show()

In [ ]:
from pyspark.sql.functions import percentile_approx

median_age = cust.select(
    percentile_approx("age", 0.5)
).first()[0]

print(median_age)

In [ ]:
cust = cust.fillna({"age": median_age})

In [ ]:
cust.show()

# **Check The Duplicate Values**

In [ ]:
arti_duplicat = arti.groupBy(arti.columns).count().filter(col("count") > 1)
arti_duplicat.show()

In [ ]:
cust_duplicat = cust.groupBy(cust.columns).count().filter(col("count") > 1)
cust_duplicat.show()

In [ ]:
tran_duplicat = tran.groupBy(tran.columns).count().filter(col("count") > 1)
tran_duplicat.count()

In [ ]:
from pyspark.sql.functions import count, avg

tran_unique = tran.groupBy('customer_id','article_id').agg(
    count("*").alias("purchase_frequency"),
    avg("price").alias("avg_price")
)
tran_unique.count()

# Feature Engnering for transection_traning.csv

In [ ]:
from pyspark.sql.functions import dayofweek, month, datediff, lag, col
from pyspark.sql.window import Window

tran = tran.withColumn("dow", dayofweek("t_dat")) \
           .withColumn("purchase_month", month("t_dat"))

w = Window.partitionBy("customer_id").orderBy("t_dat")
tran = tran.withColumn("days_since_prev_purchase",
                        datediff("t_dat", lag("t_dat").over(w)))
tran = tran.fillna({"days_since_prev_purchase": 999})

In [ ]:
tran.show(10)

In [ ]:
user_purchase_counts = tran.groupBy("customer_id").agg(count("*").alias("total_purchases"))

In [ ]:
from pyspark.sql.functions import col
active_users = user_purchase_counts.filter(col("total_purchases") >= 2)

In [ ]:
tran_filtered = tran.join(active_users, on="customer_id", how="inner")
print(f"Original transactions: {tran.count()}")
print(f"Filtered transactions: {tran_filtered.count()}")

In [ ]:
tran_filtered.show()

In [ ]:
tran_interaction = tran_filtered.groupBy("customer_id", "article_id").agg(
      count("*").alias("purchase_count")
  )

tran_interaction.show(10)

In [ ]:
user_profiles = tran_filtered.groupBy("customer_id").agg(
      avg("price").alias("avg_spending"),
      avg("days_since_prev_purchase").alias("avg_shopping_interval"),
      sum("price").alias("lifetime_value")
  )

user_profiles.show(10)

# feature engnearing for custmer.**csv**

In [ ]:
from pyspark.sql.functions import when, col

cust_clean = cust.withColumn("age_group",
      when(col("age") <= 25, "GenZ")
      .when((col("age") > 25) & (col("age") < 40), "Millennial")
      .otherwise("GenX_Plus")
)

In [ ]:
cust_clean.show(10)

## Feature eng. for artical.**csv**

In [ ]:
arti.columns

In [ ]:
arti = arti.withColumn("detail_desc", when(col("detail_desc").isNull(), "Unknown").otherwise(col("detail_desc")))

In [ ]:
arti = arti.withColumn("product_group",when(col("product_type_name").contains("T-shirt"), "T-Shirts")
      .when(col("product_type_name").contains("Dress"), "Dresses")
      .when(col("product_type_name").contains("Pants") | col("product_type_name").contains("Jeans"), "Bottoms")
      .otherwise("Other")
  )

In [ ]:
poputar_item = tran.groupBy('article_id').count().withColumnRenamed('count','popularity_score')

In [ ]:
arti = arti.join(poputar_item, on="article_id", how="left").fillna({"popularity_score": 0})

In [ ]:
arti.show()

# saving the files into my **drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
base_path = "/content/drive/MyDrive/h&m_project/"

In [ ]:
arti.coalesce(1).write.mode("overwrite").parquet(base_path + "articles_cleaned.parquet")

In [ ]:

cust_clean.coalesce(1).write.mode("overwrite").parquet(base_path + "customers_cleaned.parquet")
tran_filtered.coalesce(1).write.mode("overwrite").parquet(base_path + "transactions_cleaned.parquet")
tran_interaction.coalesce(1).write.mode("overwrite").parquet(base_path + "tran_interaction.parquet")
user_profiles.coalesce(1).write.mode("overwrite").parquet(base_path + "user_profiles.parquet")
print('data loding sucesfull')